# EngressNet training -- UNet + Engression for sea ice thickness downscaling

Notebook front-end for `sea_ice_downscaling`'s model-training modules
(`model.py`, `channels.py`, `patches.py`, `losses.py`, `training.py`, `evaluation.py`,
`land_mask.py`, `train_pipeline.py`). Same idea as the data-prep notebook: each step calls
the same functions `train_pipeline.run_training_pipeline()` calls internally, broken out so
you can inspect tensors and plots between steps.

**Before running:** put `sea_ice_downscaling/` next to this notebook (or on your
`PYTHONPATH`), and make sure `requirements.txt` is installed in this kernel.

**Important if you're loading data made by THIS package's `build_dataset.py`:** the default
channel order below is `("hi", "aice", "U", "V")` with `U`/`V` already in m/s (CESM
atmosphere winds). If you're instead loading an OLDER `X_perfmodexp.nc` from the original
5-channel notebook pipeline (`hi, Tsfc, SST, uvel, vvel`, with `uvel`/`vvel` in cm/s), change
`channel_order` and `channel_specs` in the config cell below accordingly -- see the comment
there. Getting this wrong used to fail silently (wrong channel divided by 100); it now raises
an error instead if a channel name doesn't have a matching spec.


## 1. Setup

In [ ]:
import sys
from pathlib import Path

# sys.path.insert(0, str(Path('..').resolve()))  # uncomment/adjust if needed

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sea_ice_downscaling.config import REGIONS
from sea_ice_downscaling.channels import NEW_PIPELINE_CHANNELS, LEGACY_POP_CHANNELS
from sea_ice_downscaling.train_pipeline import (
    TrainingPipelineConfig, prepare_training_data, run_training_pipeline,
)
from sea_ice_downscaling.model import build_model
from sea_ice_downscaling.training import TrainConfig, train
from sea_ice_downscaling.evaluation import (
    evaluate_ensemble, bilinear_baseline, denormalize_all, compute_metrics_table, spatial_corr,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("Device:", device)
print("GPUs:", torch.cuda.device_count())

## 2. Configure the run

`region` should match the region you used when preprocessing (`build_dataset.PipelineConfig`),
since the land mask needs to be cropped to the same bbox as your X/Y data.


In [ ]:
config = TrainingPipelineConfig(
    x_path="/glade/work/skygale/_projects/SeaIceDownscaling/data/X_perfmodexp_cons.zarr",
    y_path="/glade/work/skygale/_projects/SeaIceDownscaling/data/Y_perfmodexp.zarr",
    #
    # --- channel handling: match this to what's ACTUALLY in your saved X file ---
    # New pipeline (this package's build_dataset.py default): hi, aice, U, V (U/V already m/s)
    channel_order=("hi", "aice", "U", "V"),
    channel_specs=NEW_PIPELINE_CHANNELS,
    #
    # Old 5-channel notebook pipeline instead? Use this combination:
    # channel_order=("hi", "Tsfc", "SST", "uvel", "vvel"),
    # channel_specs=LEGACY_POP_CHANNELS,   # uvel/vvel cm/s -> m/s handled automatically
    #
    keep_channels=("hi", "U", "V"),   # matches your notebook's `X[:, :, [0,3,4], :, :]` intent
    target_channel="hi",
    #
    region=REGIONS["cambridge_bay"],   # must match what you preprocessed with
    pop_grid_hr="POP_tx0.1v2",
    weights_dir="/glade/work/skygale/_projects/SeaIceDownscaling/regrid_weights",
    #
    context_size=(16, 24),
    target_size=(8, 12),
    stride=4,
    train_frac=0.7,
    split_seed=0,
    #
    latent_channels=8,
    train_config=TrainConfig(num_epochs=20, batch_size=32, k_ensemble=6, learning_rate=1e-4),
)
config

## 3. Load, process channels, build land mask, split, normalize, extract patches

This is the slow-ish one-time step -- equivalent to your notebook's "Data preparation" cell.


In [ ]:
data = prepare_training_data(config)

print("X_train:", data["X_train"].shape)
print("Y_train:", data["Y_train"].shape)
print("M_train:", data["M_train"].shape)
print("X_test: ", data["X_test"].shape)
print("Y_test: ", data["Y_test"].shape)
print("channel_order (post-subset):", data["channel_order"])
print("target_channel_idx:", data["target_channel_idx"])

### Quick look: mask, one LR sample, one HR sample


In [ ]:
# Full-domain land mask (real lat/lon) on the left -- this is a full field, so contourf
# against hlat/hlon is valid here.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cf1 = axes[0].contourf(data["hlon"], data["hlat"], data["land_mask"][0, 0], levels=[0, 0.5, 1])
fig.colorbar(cf1, ax=axes[0]); axes[0].set_title("Land mask (full domain)")

# X_train/Y_train are PATCHES (small windows), not full domain -- plot with imshow against
# their own pixel grid rather than the full-domain lat/lon (shapes won't match contourf).
axes[1].imshow(data["X_train"][0, 0], cmap="Blues")
axes[1].set_title("X patch (normalized)")

axes[2].imshow(data["Y_train"][0, 0], cmap="Blues")
axes[2].set_title("Y patch (normalized)")

plt.tight_layout()
plt.show()

## 4. Build the model


In [ ]:
model = build_model(
    in_channels=data["X_train"].shape[1],
    latent_channels=config.latent_channels,
    device=device,
    data_parallel=False,   # set True only if you're actually running multi-GPU
)
optimizer = torch.optim.Adam(model.parameters(), lr=config.train_config.learning_rate)
print(model)

## 5. Train

Uses the Engression energy-score + land-mask loss (the loop your notebook's "Training loop
(Engression)" cell ran -- the other "Training loop (normal)" cell referenced `masked_loss` /
`base_loss`, which weren't defined anywhere in the notebook, so it isn't reproduced here).


In [ ]:
history = train(
    model, optimizer,
    data["X_train"], data["Y_train"], data["M_train"],
    config.train_config, device,
)

In [ ]:
epochs = np.arange(1, len(history.loss) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 3))

axes[0].plot(epochs, history.land, linewidth=2, color="green")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Land Loss")

axes[1].plot(epochs, history.energy, linewidth=2, color="red")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Energy Loss")

axes[2].plot(epochs, history.loss, linewidth=2, color="blue")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Training Loss")

plt.tight_layout()
plt.show()

## 6. Evaluate

Runs the model `k_eval` times per test sample (stochastic ensemble) plus once deterministically
(`z=0`), computes the bilinear baseline, converts everything back to physical units, and builds
the same comparison table your notebook printed.


In [ ]:
eval_result = evaluate_ensemble(
    model, data["X_test"], data["Y_test"],
    latent_channels=config.latent_channels, device=device,
    batch_size=16, k_eval=6,
)

baseline = bilinear_baseline(
    data["X_test"], data["Y_test"].shape[-2:], data["target_channel_idx"]
)

phys = denormalize_all(
    data["Y_test"], eval_result, data["X_test"], baseline,
    data["y_stats"], data["x_stats"], data["target_channel_idx"],
)

metrics_df = pd.DataFrame(compute_metrics_table(phys)).round(4)
print(metrics_df)

### Spatial correlation (bilinear vs. CNN)


In [ ]:
corr_cnn = spatial_corr(phys["Y_test_phys"].numpy(), phys["Y_pred_phys"].numpy())
corr_base = spatial_corr(phys["Y_test_phys"].numpy(), phys["Y_base_phys"].numpy())

print(f"\nCNN spatial correlation     : {corr_cnn:.4f}")
print(f"Bilinear spatial correlation: {corr_base:.4f}")

### Ensemble figure (low-res / bilinear / deterministic / one member / ensemble mean / truth)


In [ ]:
mem_idx = 4
num_samples = 3
idxs = np.random.choice(data["X_test"].shape[0], num_samples, replace=False)

fig, axs = plt.subplots(3, 6, figsize=(12, 4), constrained_layout=True, dpi=150)

for row, idx in enumerate(idxs):
    lowres = phys["X_test_target_phys"][idx, 0]
    bilinear = phys["Y_base_phys"][idx, 0]
    deterministic = phys["Y_pred_det_phys"][idx, 0]
    ens_full = phys["preds_all_phys"][idx]
    ens_mean = phys["Y_pred_phys"][idx, 0]
    truth = phys["Y_test_phys"][idx, 0]

    im = axs[row, 0].imshow(lowres, cmap="Blues", vmin=0, vmax=3)
    axs[row, 1].imshow(bilinear, cmap="Blues", vmin=0, vmax=3)
    axs[row, 2].imshow(deterministic, cmap="Blues", vmin=0, vmax=3)
    axs[row, 3].imshow(ens_full[mem_idx, 0], cmap="Blues", vmin=0, vmax=3)
    axs[row, 4].imshow(ens_mean, cmap="Blues", vmin=0, vmax=3)
    axs[row, 5].imshow(truth, cmap="Blues", vmin=0, vmax=3)

    for col in range(6):
        axs[row, col].set_xticks([]); axs[row, col].set_yticks([])

titles = ["Low-Res Input", "Bilinear", "Deterministic", "One Member", "Ensemble Mean", "High-Res Truth"]
for col, t in enumerate(titles):
    axs[0, col].set_title(t, fontsize=12)

cbar = fig.colorbar(im, ax=axs, aspect=15, shrink=0.99, pad=0.02)
cbar.set_label("Sea ice thickness (m)", fontsize=12)
plt.show()

## 7. Save the model checkpoint

Saves model weights plus the normalization stats and config needed to use the model later --
without the stats, predictions can't be converted back to physical units.


In [ ]:
checkpoint_path = "/glade/work/skygale/_projects/SeaIceDownscaling/checkpoints/engressnet_v1.pt"
Path(checkpoint_path).parent.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "latent_channels": config.latent_channels,
    "in_channels": data["X_train"].shape[1],
    "channel_order": data["channel_order"],
    "target_channel_idx": data["target_channel_idx"],
    "x_mean": data["x_stats"].mean,
    "x_std": data["x_stats"].std,
    "y_mean": data["y_stats"].mean,
    "y_std": data["y_stats"].std,
}, checkpoint_path)

print("Saved checkpoint to:", checkpoint_path)

## Appendix: run everything in one call


In [ ]:
# from sea_ice_downscaling.train_pipeline import run_training_pipeline
# result = run_training_pipeline(config)
# result["metrics"]